# Week 3 — Advanced RAG: Re-ranking & Query Transformation

> **Source notebook for** [`src/reranking/`](../src/reranking/).

## Learning objectives

1. Formalize the difference between Bi-Encoders and Cross-Encoders and derive the latency/quality Pareto frontier between them.
2. Implement and reason about three query-transformation strategies: rewriting, decomposition, and Hypothetical Document Embeddings (HyDE).
3. Derive the pointwise BCE and pairwise margin losses used to train cross-encoders, and implement both in NumPy.
4. Add a Cross-Encoder re-ranking stage to the Week 2 RAG pipeline and measure the lift on nDCG@k.


## 1. The Bi-Encoder bottleneck

A Bi-Encoder defines the retrieval score as a dot product of two independent embeddings:

$$
s_{\text{bi}}(q, d) = \phi(q)^\top \phi(d), \qquad \phi:\Sigma^\* \to \mathbb{R}^d.
$$

Because $\phi(d)$ depends only on $d$, all document embeddings can be precomputed and stored in an ANN index. Per-query cost is $O(d \cdot \log N)$ — fast, but with a hard accuracy ceiling: the query and document have **no opportunity to interact** during encoding.

A Cross-Encoder concatenates the pair and predicts a scalar:

$$
s_{\text{cross}}(q, d) = f_\theta\bigl([\text{CLS}]\,q\,[\text{SEP}]\,d\bigr) \in \mathbb{R}.
$$

Joint attention between query and document tokens captures fine-grained interactions (negation, anaphora, exact-match keywords) that the Bi-Encoder cannot. The cost: $\phi(d)$ cannot be precomputed, so each query requires $N$ forward passes through a Transformer.

**This is why every serious retrieval system is two-stage**:

```
Bi-Encoder    →  top 50 candidates    (cheap, parallel, recall-oriented)
Cross-Encoder →  top 5 reranked       (expensive, sequential, precision-oriented)
```


In [ ]:
# Order-of-magnitude estimate of the latency tradeoff.
N_docs        = 1_000_000
d_emb         = 384
bi_per_doc_us = 0.3          # ANN lookup dominated by graph traversal
ce_per_pair_ms = 8.0          # ms-marco-MiniLM-L-6 on CPU

bi_total_ms = N_docs * bi_per_doc_us / 1000 / 50   # HNSW prunes drastically
ce_50_ms    = 50 * ce_per_pair_ms
ce_all_ms   = N_docs * ce_per_pair_ms

print(f"Bi-Encoder over 1M docs (HNSW):           ~{bi_total_ms:6.0f} ms")
print(f"Cross-Encoder over top-50 candidates:    ~{ce_50_ms:6.0f} ms")
print(f"Cross-Encoder over 1M docs (impossible): ~{ce_all_ms:6.0f} ms ({ce_all_ms/1000/60:.0f} minutes)")


## 2. Query rewriting and decomposition

The Bi-Encoder embeds queries into a *question manifold* and documents into an *answer manifold*. The gap between the two manifolds is responsible for a large fraction of retrieval misses. Three LLM-driven mitigations:


### 2a. Rewriting

Paraphrase the user query into terminology more likely to appear in the corpus. Expand acronyms; replace colloquialisms with technical terms.

> *"can RAG be more accurate"* → *"techniques for improving the precision and recall of retrieval-augmented generation systems"*


### 2b. Decomposition

Split a multi-hop question into independent single-hop subquestions whose union is easier to retrieve for.

> *"Did the inventors of HyDE also work on BERT?"*
> → ["Who are the authors of HyDE?", "Who are the authors of BERT?"]


### 2c. Hypothetical Document Embeddings (HyDE)

Ask the LLM to *hallucinate a plausible answer document* for the query, then embed the answer instead of the query. Even with factual errors, the hallucinated answer's embedding is closer to real answers than the original query's embedding.

Formally:

$$
s_{\text{HyDE}}(q, d) = \phi\bigl(\text{LLM}(q)\bigr)^\top \phi(d).
$$

With $K$ hypotheses sampled at temperature $T>0$:

$$
\hat\phi_q = \frac{1}{K}\sum_{k=1}^K \phi\bigl(\text{LLM}_k(q)\bigr).
$$

We additionally include $\phi(q)$ in the average so off-topic hallucinations cannot destroy recall.


In [ ]:
# Demonstration with a deterministic fake LLM (no real API calls in the notebook).
# In production this is `AnthropicClient` or `OpenAIClient`.
from dataclasses import dataclass
from src.utils.llm_client import Message

class FakeLLM:
    def __init__(self, scripted):
        self.scripted = list(scripted)
    def complete(self, messages, **kw):
        return self.scripted.pop(0)

from src.reranking.query_rewriter import QueryRewriter

rewriter = QueryRewriter(FakeLLM([
    '{"rewritten": "techniques for improving precision of retrieval-augmented generation"}',
    '{"sub_questions": ["Who invented HyDE?", "Who invented BERT?"]}',
]))

print("rewrite:", rewriter.rewrite("can RAG be more accurate"))
print("decompose:", rewriter.decompose("Did the inventors of HyDE also work on BERT?"))


In [ ]:
from src.reranking.hyde import HyDE
from src.rag_engine.embeddings import SentenceEncoder

hyde = HyDE(
    llm=FakeLLM(["Sentence-Bert dense retrieval can be improved with hierarchical chunking and cross-encoder re-ranking, which jointly score query-document pairs."]),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    n_hypotheses=1,
)
qvec = hyde.embed_query("how can dense retrieval be improved")
print("HyDE query vector shape:", qvec.shape, "norm:", float(np.linalg.norm(qvec)))


## 3. Cross-Encoder training objectives

Two losses are commonly used to train re-rankers. Both are implemented in [`src/reranking/cross_encoder.py`](../src/reranking/cross_encoder.py).

### 3a. Pointwise BCE

Given relevance labels $y_i \in \{0, 1\}$ and raw scores $s_i$:

$$
\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^N \Bigl(y_i \log \sigma(s_i) + (1 - y_i)\log(1 - \sigma(s_i))\Bigr).
$$

This treats re-ranking as binary classification: *is document $i$ relevant to the query?* Easy to train, but indifferent to the *ordering* of positives among themselves.

### 3b. Pairwise margin (hinge)

Given a positive $(q, d^+)$ and a negative $(q, d^-)$, enforce a margin:

$$
\mathcal{L}_{\text{margin}} = \frac{1}{N}\sum_{i=1}^N \max\bigl(0,\, m - (s_i^+ - s_i^-)\bigr).
$$

This directly optimizes the *ordering*, which is what retrieval cares about, but requires constructing $(q, d^+, d^-)$ triples — usually via hard-negative mining over the Bi-Encoder's top-k.


In [ ]:
import numpy as np
from src.reranking.cross_encoder import pointwise_bce_loss, pairwise_margin_loss

# Pointwise: perfect predictions → ~0 loss.
scores = np.array([ 5.0,  5.0, -5.0, -5.0])
labels = np.array([ 1.0,  1.0,  0.0,  0.0])
print(f"BCE (perfect):   {pointwise_bce_loss(scores, labels):.6f}")

# Pointwise: inverted predictions → large loss.
print(f"BCE (inverted):  {pointwise_bce_loss(-scores, labels):.4f}")

# Pairwise: positives clearly above negatives → 0 loss.
pos = np.array([3.0, 3.0]); neg = np.array([0.0, 0.0])
print(f"Margin (sep'd):  {pairwise_margin_loss(pos, neg, margin=1.0):.4f}")

# Pairwise: negatives above positives → positive loss.
print(f"Margin (mixed):  {pairwise_margin_loss(neg, pos, margin=1.0):.4f}")


## 4. End-to-end: re-ranking lift

We assemble the full pipeline: HierarchicalRAG (Bi-Encoder retrieval) → CrossEncoderReranker (precision re-rank).


In [ ]:
from src.rag_engine.chunking import HierarchicalChunker
from src.rag_engine.pipeline import HierarchicalRAG
from src.rag_engine.vector_stores import build_store
from src.reranking.cross_encoder import CrossEncoderReranker
from src.reranking.pipeline import RerankingPipeline
import tempfile

persist = tempfile.mkdtemp()
rag = HierarchicalRAG(
    chunker=HierarchicalChunker(parent_chars=400, child_chars=180),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    child_store=build_store("chroma", collection="rerank_demo", persist_dir=persist),
    parent_lookup={},
)

corpus = [
    ("hyde",     "HyDE generates a hypothetical answer document and embeds it for retrieval. The synthetic document lies closer to real answer documents than the original query does."),
    ("rerank",   "Cross-encoders score query-document pairs jointly via the transformer attention mechanism. They are slower than bi-encoders but substantially more accurate."),
    ("chunking", "Hierarchical chunking splits long documents into parent and child segments. Children are indexed for retrieval; parents are returned for context."),
    ("hnsw",     "HNSW is a graph-based approximate nearest neighbour algorithm. It maintains a hierarchy of small-world graphs and achieves logarithmic expected search time."),
    ("react",    "ReAct interleaves Thought, Action, and Observation steps. The language model verbalizes a plan, emits a tool call, and reads back the result."),
]
for doc_id, text in corpus:
    rag.ingest_document(text, metadata={"doc_id": doc_id, "title": doc_id})

reranker = RerankingPipeline(reranker=CrossEncoderReranker(), candidate_k=5)

query = "how do cross-encoders compare to bi-encoders?"
candidates = rag.retrieve(query, k=5)
print("BI-ENCODER TOP 5:")
for c in candidates:
    print(f"  [{c.score:+.3f}] {c.metadata['doc_id']}")

reranked = reranker.rerank(query, candidates, k=3)
print("\nCROSS-ENCODER TOP 3:")
for r in reranked:
    print(f"  [{r.cross_encoder_score:+.3f}] {r.metadata['doc_id']}")


**What just happened.** The Bi-Encoder returned five candidates ordered by their cosine similarity. The Cross-Encoder *re-scored* them by jointly attending to the query and each document, producing a different (and usually better) ordering. On the BEIR benchmark slice in `docs/benchmarks.md` this lifts nDCG@10 from 0.58 to 0.74 — a ~28% relative improvement at the cost of ~85 ms additional latency.


## 5. The Cohere swap

The same `Reranker` Protocol accepts a hosted re-ranker. To swap in Cohere:

```yaml
# configs/reranker_cohere.yaml
reranker:
  backend: cohere
  model: rerank-english-v3.0
  candidate_k: 50
```

```python
RerankingPipeline.from_config("configs/reranker_cohere.yaml")
```

The pipeline code is unchanged. This is the dividend of programming to the interface.


## 6. Exercises

1. **Hard-negative mining.** Implement a script that uses the Bi-Encoder's top-k to mine *hard negatives* for cross-encoder fine-tuning. Compare in-batch random negatives vs. hard negatives on margin-loss training curves.
2. **HyDE ablation.** Build a mini-eval set of 50 queries. Compare retrieval recall@10 with: (a) plain query, (b) rewritten query, (c) HyDE ($K=1$), (d) HyDE ($K=4$). Which transformation pays for the latency it costs?
3. **Distillation.** Train a small (3-layer) cross-encoder to mimic the scores of a large one using MSE on logits. How much accuracy is preserved? How much latency is saved?


## 7. Take-aways

- The Bi/Cross-Encoder duality is the single most important architectural pattern in modern retrieval. Two stages, optimized for two different objectives.
- Query transformation (rewriting, decomposition, HyDE) attacks the question/answer manifold gap. HyDE is the largest single improvement available for free at inference time.
- The choice between pointwise and pairwise loss reflects what the system is being optimized for: *classification* vs. *ordering*. Re-ranking is fundamentally an ordering task.
- Treat the re-ranker as an interface, not an implementation. A from-scratch CrossEncoder, a Hugging Face model, and the Cohere API all satisfy the same Protocol.

➡ Next week we shift from *retrieval* to *agency*: the ReAct loop that lets an LLM use these retrieval tools deliberately.
